# NegotiateEnv Training - Final Version

**Team**: Madhavi Gulavani, Mayuka Reddy & Kushal Adhyaru  
**Environment**: https://kushaladhyaru-negotiate-env.hf.space  
**Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

---

## Configuration

- **Training**: Local environment (no HTTP calls)
- **Baseline episodes**: 200
- **Training episodes**: 500 (better data quality)
- **Max turns**: 10 per episode
- **Method**: HuggingFace TRL with LoRA
- **Baseline**: 0.4833
- **Target**: 0.55-0.60
- **Time**: ~20-30 minutes on H100, ~2-3 hours on T4

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone Repository (Fresh Start)

In [ ]:
# Clean start - remove old directory
!rm -rf openEnv-negotiateEnv

# Clone fresh from GitHub
!git clone https://github.com/MadhaviSG/openEnv-negotiateEnv.git

# Change to repo directory
%cd openEnv-negotiateEnv

# Verify critical files exist
!ls -la train_local_fast.py evaluate_local.py

## 3. Install Dependencies

In [ ]:
# Install the negotiate_env package
!pip install -q -e .

# Install training dependencies
!pip install -q transformers accelerate peft datasets trl matplotlib numpy

print("\nAll dependencies installed!")

## 4. Run Baseline Evaluation (200 Episodes)

This establishes your baseline performance using a rule-based agent.

In [ ]:
print("Running baseline evaluation (200 episodes)...")
print("Using local environment (no HTTP calls)\n")
print("="*60)

!python evaluate_local.py --agent rule --episodes 200

print("\n" + "="*60)
print("Baseline evaluation complete!")

## 5. Run Training (500 Episodes)

**Configuration:**
- Model: Qwen/Qwen2.5-1.5B-Instruct
- Episodes: 500 (better data quality)
- Max turns per episode: 10
- LoRA rank: 16
- Epochs: 3

**Expected Time:**
- H100: ~20-30 minutes
- T4: ~2-3 hours

**Expected Data Collection Reward: ~0.11**  
**Baseline to Beat: 0.4833**  
**Target After Training: 0.55-0.60 (15-25% improvement)**

In [ ]:
print("Starting local training with TRL...")
print("Training on 500 episodes (local environment)")
print("")
print("Expected time:")
print("  - H100: ~20-30 minutes")
print("  - T4: ~2-3 hours")
print("")
print("="*60)

!python train_local_fast.py \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-trl-output \
    --num-episodes 500 \
    --max-turns 10

print("\n" + "="*60)
print("Training complete!")
print("Model saved to: negotiate-trl-output/")
print("="*60)

## 6. Login to HuggingFace

Uses your `Huggingface_Token` from Google Colab secrets.

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

print("Logging in to HuggingFace...\n")

try:
    hf_token = userdata.get('Huggingface_Token')
    login(token=hf_token)
    print("Successfully logged in using Colab secrets!")
except Exception as e:
    print(f"[ERROR] {e}")
    print("\nFalling back to interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()

## 7. Upload Model to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
output_dir = "negotiate-trl-output"
repo_id = "KushalAdhyaru/negotiate-env-qwen-500ep"

if os.path.exists(output_dir):
    print(f"Uploading {output_dir} to {repo_id}...\n")
    print("="*60)
    
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print("\n" + "="*60)
        print("Model uploaded successfully!")
        print(f"\nView at: https://huggingface.co/{repo_id}")
        print("="*60)
    except Exception as e:
        print(f"\n[ERROR] {e}")
        print("You can manually upload later.")
else:
    print(f"[ERROR] Directory not found: {output_dir}")

## 8. Download Results (Optional)

In [ ]:
print("Creating results archive...\n")

!zip -r training_results_fast.zip \
    negotiate-trl-output/ \
    *.log 2>/dev/null || true

print("\nResults saved to training_results_fast.zip")
print("Download from the file browser (left sidebar)")

## Summary

### What You Accomplished:

1. Baseline evaluation: 200 episodes (local environment)
2. Training: 500 episodes with TRL + LoRA
3. Model uploaded to HuggingFace Hub

### Hackathon Submission Links:

- **Environment**: https://kushaladhyaru-negotiate-env.hf.space
- **Dataset**: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset
- **Trained Model**: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-500ep
- **Code Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

Good luck with your hackathon submission!